# Считывание Данных

In [67]:
import pandas as pd 
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import optuna
import numpy as np
import warnings
from datetime import datetime
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance
import optuna.visualization as vis
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

In [68]:
drive_files = path + '/*.parquet'
drives_needed_cols = ["request_datetime", "PULocationID"]
drives = pl.scan_parquet(drive_files).select([pl.col(x) for x in drives_needed_cols])
print(drives.head(5).collect())

shape: (5, 2)
┌─────────────────────┬──────────────┐
│ request_datetime    ┆ PULocationID │
│ ---                 ┆ ---          │
│ datetime[ns]        ┆ i64          │
╞═════════════════════╪══════════════╡
│ 2021-01-01 00:28:09 ┆ 230          │
│ 2021-01-01 00:45:56 ┆ 152          │
│ 2021-01-01 00:21:15 ┆ 233          │
│ 2021-01-01 00:39:12 ┆ 142          │
│ 2021-01-01 00:46:11 ┆ 143          │
└─────────────────────┴──────────────┘


In [69]:
meteodata_files = path + '/nyc 2021-01-01 to 2021-12-31.csv'
meteo_needed_cols = [
    'datetime', 'temp', 'feelslike', 'dew', 'humidity', 'precip', 'precipprob',
    'preciptype', 'snow', 'snowdepth', 'windgust','windspeed', 'winddir',
    'sealevelpressure', 'cloudcover', 'visibility', 'uvindex'
]
meteodata = pl.scan_csv(meteodata_files).select([pl.col(x) for x in meteo_needed_cols])
print(meteodata.head(5).collect())

shape: (5, 17)
┌────────────┬──────┬───────────┬──────┬───┬──────────────────┬────────────┬────────────┬─────────┐
│ datetime   ┆ temp ┆ feelslike ┆ dew  ┆ … ┆ sealevelpressure ┆ cloudcover ┆ visibility ┆ uvindex │
│ ---        ┆ ---  ┆ ---       ┆ ---  ┆   ┆ ---              ┆ ---        ┆ ---        ┆ ---     │
│ str        ┆ f64  ┆ f64       ┆ f64  ┆   ┆ f64              ┆ f64        ┆ f64        ┆ i64     │
╞════════════╪══════╪═══════════╪══════╪═══╪══════════════════╪════════════╪════════════╪═════════╡
│ 2021-01-01 ┆ 2.5  ┆ -0.2      ┆ -3.0 ┆ … ┆ 1028.9           ┆ 50.6       ┆ 14.0       ┆ 3       │
│ 2021-01-02 ┆ 5.8  ┆ 3.6       ┆ 1.2  ┆ … ┆ 1012.4           ┆ 63.9       ┆ 12.2       ┆ 5       │
│ 2021-01-03 ┆ 2.5  ┆ -1.6      ┆ -0.5 ┆ … ┆ 1017.0           ┆ 81.5       ┆ 13.2       ┆ 1       │
│ 2021-01-04 ┆ 3.6  ┆ 1.1       ┆ -0.2 ┆ … ┆ 1014.6           ┆ 89.3       ┆ 15.6       ┆ 4       │
│ 2021-01-05 ┆ 3.8  ┆ 1.3       ┆ -1.5 ┆ … ┆ 1013.1           ┆ 98.8       ┆ 16.0    

In [70]:
import kagglehub

path = kagglehub.dataset_download("shuhengmo/uber-nyc-forhire-vehicles-trip-data-2021")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/shuhengmo/uber-nyc-forhire-vehicles-trip-data-2021


# Первичная обработка данных

## Сжимаем датасет до часовой точности, суммируя поездки по локациям

In [71]:
start_date = datetime(2021, 1, 1)
second_week = datetime(2021, 1, 7)

drives_lazy = (
    drives
    .filter(pl.col("request_datetime") >= start_date)
    .group_by([
        pl.col("request_datetime").dt.truncate("1h").alias("hour_bucket"),
        "PULocationID"
    ])
    .agg(pl.len().alias("trips_count"))

    .sort(["PULocationID", "hour_bucket"])
)

print(drives_lazy.collect(engine="streaming").head(5))

shape: (5, 3)
┌─────────────────────┬──────────────┬─────────────┐
│ hour_bucket         ┆ PULocationID ┆ trips_count │
│ ---                 ┆ ---          ┆ ---         │
│ datetime[ns]        ┆ i64          ┆ u32         │
╞═════════════════════╪══════════════╪═════════════╡
│ 2021-01-01 05:00:00 ┆ 1            ┆ 1           │
│ 2021-01-01 07:00:00 ┆ 1            ┆ 1           │
│ 2021-01-01 10:00:00 ┆ 1            ┆ 1           │
│ 2021-01-01 12:00:00 ┆ 1            ┆ 1           │
│ 2021-01-01 16:00:00 ┆ 1            ┆ 1           │
└─────────────────────┴──────────────┴─────────────┘


## Заполняем временной ряд полностью, добавляя записи даже в которые не произошло ни одной поездки

In [72]:
end_date = drives_lazy.select(pl.col("hour_bucket").max()).collect().item()
target_dtype = drives_lazy.collect_schema()["hour_bucket"]
time_grid = pl.datetime_range(
    start=start_date, 
    end=end_date, 
    interval="1h", 
    eager=True
).alias("hour_bucket").to_frame().cast({"hour_bucket": target_dtype}).lazy()

locations = drives.select(pl.col("PULocationID").unique())

grid = time_grid.join(locations, how="cross")

full_drives_lazy = grid.join(
    drives_lazy, 
    on=["hour_bucket", "PULocationID"], 
    how="left"
).with_columns(
    pl.col("trips_count").fill_null(0)
)

full_drives_lazy = full_drives_lazy.sort(["PULocationID", "hour_bucket"])

print(full_drives_lazy.collect(engine="streaming").head(6))

shape: (6, 3)
┌─────────────────────┬──────────────┬─────────────┐
│ hour_bucket         ┆ PULocationID ┆ trips_count │
│ ---                 ┆ ---          ┆ ---         │
│ datetime[ns]        ┆ i64          ┆ u32         │
╞═════════════════════╪══════════════╪═════════════╡
│ 2021-01-01 00:00:00 ┆ 1            ┆ 0           │
│ 2021-01-01 01:00:00 ┆ 1            ┆ 0           │
│ 2021-01-01 02:00:00 ┆ 1            ┆ 0           │
│ 2021-01-01 03:00:00 ┆ 1            ┆ 0           │
│ 2021-01-01 04:00:00 ┆ 1            ┆ 0           │
│ 2021-01-01 05:00:00 ┆ 1            ┆ 1           │
└─────────────────────┴──────────────┴─────────────┘


## Разделение request_datetime на hour, dayofweek, month, day

In [73]:
full_drives_lazy = (
    full_drives_lazy
    .with_columns([
        pl.col("hour_bucket").dt.month().alias("month"),
        pl.col("hour_bucket").dt.day().alias("day"),
        pl.col("hour_bucket").dt.weekday().alias("dayofweek"),
        pl.col("hour_bucket").dt.hour().alias("hour"),
        pl.col("hour_bucket").dt.date().alias("date"),
    ])

    .with_columns([
        pl.col("trips_count").shift(1).over("PULocationID").fill_null(0).alias("trips_lag_1h"),
        pl.col("trips_count").shift(24).over("PULocationID").fill_null(0).alias("trips_lag_day"),
        pl.col("trips_count").shift(24 * 7).over("PULocationID").fill_null(0).alias("trips_lag_week"),
    ])

    .with_columns([
        pl.col("trips_count").rolling_mean(24, min_samples=1).over("PULocationID").alias("roll_mean_24h"),
        pl.col("trips_count").rolling_std(24, min_samples=1).over("PULocationID").alias("roll_std_24h"),
        pl.col("trips_count").rolling_mean(24*7, min_samples=1).over("PULocationID").alias("roll_mean_week"),
    ])

    .with_columns([
        np.sin(2 * np.pi * pl.col("hour") / 24).alias("hour_sin"),
        np.cos(2 * np.pi * pl.col("hour") / 24).alias("hour_cos"),
    ])

    .filter(pl.col("hour_bucket") >= second_week)   
)

zone_mean = (
    full_drives_lazy.group_by("PULocationID")
    .agg(pl.col("trips_count").mean().alias("zone_avg_trips"))
)

full_drives_lazy = full_drives_lazy.join(zone_mean, on="PULocationID", how="left")

print(full_drives_lazy.head(10).collect())

shape: (10, 17)
┌────────────┬────────────┬────────────┬───────┬───┬────────────┬──────────┬───────────┬───────────┐
│ hour_bucke ┆ PULocation ┆ trips_coun ┆ month ┆ … ┆ roll_mean_ ┆ hour_sin ┆ hour_cos  ┆ zone_avg_ │
│ t          ┆ ID         ┆ t          ┆ ---   ┆   ┆ week       ┆ ---      ┆ ---       ┆ trips     │
│ ---        ┆ ---        ┆ ---        ┆ i8    ┆   ┆ ---        ┆ f64      ┆ f64       ┆ ---       │
│ datetime[n ┆ i64        ┆ u32        ┆       ┆   ┆ f64        ┆          ┆           ┆ f64       │
│ s]         ┆            ┆            ┆       ┆   ┆            ┆          ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════╪═══╪════════════╪══════════╪═══════════╪═══════════╡
│ 2021-01-07 ┆ 1          ┆ 0          ┆ 1     ┆ … ┆ 0.889655   ┆ 0.0      ┆ 1.0       ┆ 0.269383  │
│ 00:00:00   ┆            ┆            ┆       ┆   ┆            ┆          ┆           ┆           │
│ 2021-01-07 ┆ 1          ┆ 1          ┆ 1     ┆ … ┆ 0.890411   ┆ 0.258819 

## Преобразование временных данных в метеоданных

In [74]:
meteodata_lazy = (
    meteodata.with_columns(
        pl.col('datetime').str.to_date().alias('date')
    )
    .drop('datetime')
)

print(meteodata_lazy.head(5).collect())

shape: (5, 17)
┌──────┬───────────┬──────┬──────────┬───┬────────────┬────────────┬─────────┬────────────┐
│ temp ┆ feelslike ┆ dew  ┆ humidity ┆ … ┆ cloudcover ┆ visibility ┆ uvindex ┆ date       │
│ ---  ┆ ---       ┆ ---  ┆ ---      ┆   ┆ ---        ┆ ---        ┆ ---     ┆ ---        │
│ f64  ┆ f64       ┆ f64  ┆ f64      ┆   ┆ f64        ┆ f64        ┆ i64     ┆ date       │
╞══════╪═══════════╪══════╪══════════╪═══╪════════════╪════════════╪═════════╪════════════╡
│ 2.5  ┆ -0.2      ┆ -3.0 ┆ 67.8     ┆ … ┆ 50.6       ┆ 14.0       ┆ 3       ┆ 2021-01-01 │
│ 5.8  ┆ 3.6       ┆ 1.2  ┆ 74.0     ┆ … ┆ 63.9       ┆ 12.2       ┆ 5       ┆ 2021-01-02 │
│ 2.5  ┆ -1.6      ┆ -0.5 ┆ 80.7     ┆ … ┆ 81.5       ┆ 13.2       ┆ 1       ┆ 2021-01-03 │
│ 3.6  ┆ 1.1       ┆ -0.2 ┆ 76.6     ┆ … ┆ 89.3       ┆ 15.6       ┆ 4       ┆ 2021-01-04 │
│ 3.8  ┆ 1.3       ┆ -1.5 ┆ 68.7     ┆ … ┆ 98.8       ┆ 16.0       ┆ 2       ┆ 2021-01-05 │
└──────┴───────────┴──────┴──────────┴───┴────────────┴──────────

## Join метеоданных 

In [97]:
result_lazy = full_drives_lazy.join(meteodata_lazy, on="date", how="inner")
result_lazy = result_lazy.drop("date")
result_df = result_lazy.collect()
result_df.head(25)

hour_bucket,PULocationID,trips_count,month,day,dayofweek,hour,trips_lag_1h,trips_lag_day,trips_lag_week,roll_mean_24h,roll_std_24h,roll_mean_week,hour_sin,hour_cos,zone_avg_trips,temp,feelslike,dew,humidity,precip,precipprob,preciptype,snow,snowdepth,windgust,windspeed,winddir,sealevelpressure,cloudcover,visibility,uvindex
datetime[ns],i64,u32,i8,i8,i8,i8,u32,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,i64
2021-01-07 00:00:00,1,0,1,7,4,0,1,0,0,0.666667,0.816497,0.889655,0.0,1.0,0.269383,1.9,-2.1,-6.4,54.9,0.0,0,null,0.0,0.0,40.7,22.8,322.7,1018.5,7.6,16.0,5
2021-01-07 01:00:00,1,1,1,7,4,1,0,0,0,0.708333,0.80645,0.890411,0.258819,0.965926,0.269383,1.9,-2.1,-6.4,54.9,0.0,0,null,0.0,0.0,40.7,22.8,322.7,1018.5,7.6,16.0,5
2021-01-07 02:00:00,1,0,1,7,4,2,1,0,0,0.708333,0.80645,0.884354,0.5,0.866025,0.269383,1.9,-2.1,-6.4,54.9,0.0,0,null,0.0,0.0,40.7,22.8,322.7,1018.5,7.6,16.0,5
2021-01-07 03:00:00,1,0,1,7,4,3,0,0,0,0.708333,0.80645,0.878378,0.707107,0.707107,0.269383,1.9,-2.1,-6.4,54.9,0.0,0,null,0.0,0.0,40.7,22.8,322.7,1018.5,7.6,16.0,5
2021-01-07 04:00:00,1,0,1,7,4,4,0,0,0,0.708333,0.80645,0.872483,0.866025,0.5,0.269383,1.9,-2.1,-6.4,54.9,0.0,0,null,0.0,0.0,40.7,22.8,322.7,1018.5,7.6,16.0,5
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2021-01-07 20:00:00,1,1,1,7,4,20,1,0,0,0.541667,0.658005,0.848485,-0.866025,0.5,0.269383,1.9,-2.1,-6.4,54.9,0.0,0,null,0.0,0.0,40.7,22.8,322.7,1018.5,7.6,16.0,5
2021-01-07 21:00:00,1,1,1,7,4,21,1,0,0,0.583333,0.653863,0.849398,-0.707107,0.707107,0.269383,1.9,-2.1,-6.4,54.9,0.0,0,null,0.0,0.0,40.7,22.8,322.7,1018.5,7.6,16.0,5
2021-01-07 22:00:00,1,2,1,7,4,22,1,1,0,0.625,0.710939,0.856287,-0.5,0.866025,0.269383,1.9,-2.1,-6.4,54.9,0.0,0,null,0.0,0.0,40.7,22.8,322.7,1018.5,7.6,16.0,5


## Заполнение пропусков и кодирование типа осадков

In [98]:
result_df = (
    result_df.
    with_columns([
        pl.col("preciptype").str.contains("rain").fill_null(False).cast(pl.Int8).alias("is_rain"),
        pl.col("preciptype").str.contains("snow").fill_null(False).cast(pl.Int8).alias("is_snow")
    ])
    .drop("preciptype")
)

global_median = result_df.select(pl.col("windgust").median()).to_numpy()[0,0]

result_df = result_df.with_columns([
    pl.when(pl.col("windgust").is_null()).then(1).otherwise(0).alias("windgust_missing"),
    pl.col("windgust")
      .fill_null(pl.col("windspeed"))
      .fill_null(global_median)
      .alias("windgust")
])

# result_df = result_df.drop_nulls()

## Аналитика

In [101]:
cols_for_base_analysis = [
    "feelslike", "dew", "humidity",
    "snowdepth", "winddir", "sealevelpressure",
    "cloudcover", "visibility", "uvindex"
]

base_atributes = ["temp", "precip", "precipprob", "windgust", "windspeed"]

target = "trips_count"


description = {
    "hour_bucket": "Дата и время с точностью до часа",
    "PULocationID": "Зона такси TLC, в которой начался рейс",
    "trips_count": "Кол-во поездок за час из определенной зоны",
    "trips_lag_1h": "Кол-во поездок за час из определенной зоны за час до текущего времени",
    "trips_lag_day": "Кол-во поездок за сутки из определенной зоны за час до текущего времени",
    "trips_lag_week": "Кол-во поездок за неделю из определенной зоны за час до текущего времени",
    "hour": "Час вызова такси",
    "dayofweek": "День недели вызова такси(1-7)",
    "month": "Месяц вызова такси",
    "day": "День вызова такси",
    "temp": "Температура воздуха",
    "feelslike": "Ощущаемая температура воздуха",
    "dew": "Точка росы",
    "humidity": "Влажность",
    "precip": "Осадки?",
    "precipprob": "Вероятность осадков",
    "snowdepth": "Глубина снега",
    "windgust": "Порывы ветра",
    "windspeed": "Скорость ветра",
    "winddir": "Направление ветра",
    "sealevelpressure": "Атмосферное давление, приведенное к среднему уровню моря ",
    "cloudcover": "Облачность",
    "visibility": "Видимость",
    "uvindex": "УФ-Индекс"
}

## Анализ выбросов. Скользящее окно

In [102]:
window_size = 24
threshold = 3
df_processed = result_df.with_columns([
    (pl.col(target).cast(pl.Float64).rolling_median(window_size=window_size, center=True)).alias("med"),
    (pl.col(target).cast(pl.Float64).rolling_std(window_size=window_size, center=True)).alias("std")
]).with_columns([
    (pl.col("med") + threshold * pl.col("std")).alias("upper"),
    (pl.col("med") - threshold * pl.col("std")).clip(lower_bound=0).alias("lower") 
]).with_columns([
    pl.col(target).cast(pl.Float64)
      .clip(pl.col("lower"), pl.col("upper"))
      .round(0)
      .cast(pl.UInt32)
      .alias("value_cleaned")
])

result_df = (
    df_processed.
    drop(["med", "std", "upper", "lower", "value_cleaned"])
)
result_df.head(43)

hour_bucket,PULocationID,trips_count,month,day,dayofweek,hour,trips_lag_1h,trips_lag_day,trips_lag_week,roll_mean_24h,roll_std_24h,roll_mean_week,hour_sin,hour_cos,zone_avg_trips,temp,feelslike,dew,humidity,precip,precipprob,snow,snowdepth,windgust,windspeed,winddir,sealevelpressure,cloudcover,visibility,uvindex,is_rain,is_snow,windgust_missing
datetime[ns],i64,u32,i8,i8,i8,i8,u32,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i8,i8,i32
2021-01-07 00:00:00,1,0,1,7,4,0,1,0,0,0.666667,0.816497,0.889655,0.0,1.0,0.269383,1.9,-2.1,-6.4,54.9,0.0,0,0.0,0.0,40.7,22.8,322.7,1018.5,7.6,16.0,5,0,0,0
2021-01-07 01:00:00,1,1,1,7,4,1,0,0,0,0.708333,0.80645,0.890411,0.258819,0.965926,0.269383,1.9,-2.1,-6.4,54.9,0.0,0,0.0,0.0,40.7,22.8,322.7,1018.5,7.6,16.0,5,0,0,0
2021-01-07 02:00:00,1,0,1,7,4,2,1,0,0,0.708333,0.80645,0.884354,0.5,0.866025,0.269383,1.9,-2.1,-6.4,54.9,0.0,0,0.0,0.0,40.7,22.8,322.7,1018.5,7.6,16.0,5,0,0,0
2021-01-07 03:00:00,1,0,1,7,4,3,0,0,0,0.708333,0.80645,0.878378,0.707107,0.707107,0.269383,1.9,-2.1,-6.4,54.9,0.0,0,0.0,0.0,40.7,22.8,322.7,1018.5,7.6,16.0,5,0,0,0
2021-01-07 04:00:00,1,0,1,7,4,4,0,0,0,0.708333,0.80645,0.872483,0.866025,0.5,0.269383,1.9,-2.1,-6.4,54.9,0.0,0,0.0,0.0,40.7,22.8,322.7,1018.5,7.6,16.0,5,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2021-01-08 14:00:00,1,0,1,8,5,14,0,2,0,0.416667,0.653863,0.839286,-0.5,-0.866025,0.269383,0.2,-3.7,-6.9,59.2,0.0,0,0.0,0.0,43.5,20.5,242.7,1015.3,25.7,16.0,3,0,0,0
2021-01-08 15:00:00,1,1,1,8,5,15,0,0,0,0.458333,0.658005,0.845238,-0.707107,-0.707107,0.269383,0.2,-3.7,-6.9,59.2,0.0,0,0.0,0.0,43.5,20.5,242.7,1015.3,25.7,16.0,3,0,0,0
2021-01-08 16:00:00,1,1,1,8,5,16,1,2,1,0.416667,0.583592,0.845238,-0.866025,-0.5,0.269383,0.2,-3.7,-6.9,59.2,0.0,0,0.0,0.0,43.5,20.5,242.7,1015.3,25.7,16.0,3,0,0,0


In [103]:
null_counts = result_df.null_count()
has_nulls = null_counts.sum_horizontal()[0] > 0
print(has_nulls)

False


In [40]:
stats = []

for col in cols_for_base_analysis:
    stats.extend([
        pl.col(col).min().alias(f'{col}_min'),
        pl.col(col).max().alias(f'{col}_max'),
        pl.col(col).mean().alias(f'{col}_mean'),
        pl.col(col).median().alias(f'{col}_median'),
        pl.col(col).std().alias(f'{col}_std')
    ])

results = result_df.select(stats)
for col in cols_for_base_analysis:
    print(f"--- {col} ---")
    print(f"Описание: {description[col]}")
    print(f"Мин: {results[0, f'{col}_min']}")
    print(f"Макс: {results[0, f'{col}_max']}")
    print(f"Среднее: {results[0, f'{col}_mean']:.2f}")
    print(f"Медиана: {results[0, f'{col}_median']}")
    print(f"Ст. отк.: {results[0, f'{col}_std']:.2f}\n")

--- feelslike ---
Описание: Ощущаемая температура воздуха
Мин: -14.1
Макс: 33.6
Среднее: 13.03
Медиана: 14.3
Ст. отк.: 11.13

--- dew ---
Описание: Точка росы
Мин: -19.7
Макс: 22.3
Среднее: 5.46
Медиана: 5.8
Ст. отк.: 10.72

--- humidity ---
Описание: Влажность
Мин: 22.6
Макс: 93.0
Среднее: 59.06
Медиана: 59.6
Ст. отк.: 15.65

--- snowdepth ---
Описание: Глубина снега
Мин: 0.0
Макс: 31.7
Среднее: 1.80
Медиана: 0.0
Ст. отк.: 5.43

--- winddir ---
Описание: Направление ветра
Мин: 27.6
Макс: 322.7
Среднее: 212.37
Медиана: 232.8
Ст. отк.: 77.65

--- sealevelpressure ---
Описание: Атмосферное давление, приведенное к среднему уровню моря 
Мин: 996.9
Макс: 1034.1
Среднее: 1015.73
Медиана: 1015.7
Ст. отк.: 6.69

--- cloudcover ---
Описание: Облачность
Мин: 5.9
Макс: 100.0
Среднее: 52.36
Медиана: 52.1
Ст. отк.: 27.37

--- visibility ---
Описание: Видимость
Мин: 1.2
Макс: 16.0
Среднее: 14.79
Медиана: 16.0
Ст. отк.: 2.16

--- uvindex ---
Описание: УФ-Индекс
Мин: 0
Макс: 10
Среднее: 6.22
Медиана: 

## Распределение основных признаков

In [41]:
test_flag = True

In [48]:
if not test_flag:
    stats = []

    for col in base_atributes:
        stats.extend([
            pl.col(col).min().alias(f'{col}_min'),
            pl.col(col).max().alias(f'{col}_max'),
            pl.col(col).mean().alias(f'{col}_mean'),
            pl.col(col).median().alias(f'{col}_median'),
            pl.col(col).std().alias(f'{col}_std')
        ])

    results = result_df.select(stats)
    for col in base_atributes:
        print(f"--- {col} ---")
        print(f"Описание: {description[col]}")
        print(f"Мин: {results[0, f'{col}_min']}")
        print(f"Макс: {results[0, f'{col}_max']}")
        print(f"Среднее: {results[0, f'{col}_mean']:.2f}")
        print(f"Медиана: {results[0, f'{col}_median']}")
        print(f"Ст. отк.: {results[0, f'{col}_std']:.2f}\n")

        plt.figure(figsize=(10, 6))
        sns.histplot(result_df[col], bins=100, kde=True, color='blue')

        plt.xlabel(description[col])
        plt.ylabel('Частота')
        plt.xlim(right=result_df[col].quantile(0.99))
        plt.show()

## Распределение Целевой переменной

In [50]:
if not test_flag:
    stats = []
    stats.extend([
        pl.col(target).min().alias(f'{target}_min'),
        pl.col(target).max().alias(f'{target}_max'),
        pl.col(target).mean().alias(f'{target}_mean'),
        pl.col(target).median().alias(f'{target}_median'),
        pl.col(target).std().alias(f'{target}_std')
    ])
    results = result_df.select(stats)
    print(f"--- {target} ---")
    print(f"Описание: {description[target]}")
    print(f"Мин: {results[0, f'{target}_min']}")
    print(f"Макс: {results[0, f'{target}_max']}")
    print(f"Среднее: {results[0, f'{target}_mean']:.2f}")
    print(f"Медиана: {results[0, f'{target}_median']}")
    print(f"Ст. отк.: {results[0, f'{target}_std']:.2f}\n")

    plt.figure(figsize=(10, 6))

    sns.histplot(result_df['trips_count'], bins=100, kde=True, color='blue')

    plt.title('Распределение количества поездок в час')
    plt.xlabel('Количество поездок')
    plt.ylabel('Частота')
    plt.show()

## Разделяем датасет на выборку для кросс-валидации и тестовую

In [104]:
result_df = result_df.sort("hour_bucket")

train_cv_finish = datetime(2021, 11, 30)
train_cv = result_df.filter(pl.col("hour_bucket") < train_cv_finish)
test = result_df.filter(pl.col("hour_bucket") >= train_cv_finish)

print(f"Train: {train_cv.shape[0]} строк")
print(f"Test:  {test.shape[0]} строк")

Train: 2064024 строк
Test:  201984 строк


## Масштабируем метеоданные

In [35]:
from sklearn.preprocessing import MinMaxScaler

numeric_cols = [
     'temp', 'feelslike', 'dew', 'humidity', 'precip', 'precipprob',
    'snowdepth', 'windgust','windspeed', 'winddir',
    'sealevelpressure', 'cloudcover', 'visibility', 'uvindex'
]

scaler = MinMaxScaler(feature_range=(0, 1))

train_scaled = scaler.fit_transform(train_cv[numeric_cols])
test_scaled = scaler.transform(test[numeric_cols])

# Построение моделей

### Подготовка данных для обучения моделей. Expanding Window CV

In [106]:
features = [col for col in train_cv.columns if col not in [target, "hour_bucket"]]

X_np = train_cv.select(features).to_pandas().values
y_np = train_cv.select(target).to_pandas().values.ravel()

tscv = TimeSeriesSplit(n_splits=3, test_size=24*31)
warnings.filterwarnings("ignore", message="X does not have valid feature names")

## Алгоритм LightGBM

### Отбор признаков с помощью Permitation Importance

In [108]:
params_bstg = {
    "boosting_type": "gbdt",
    "objective": "regression_l1",
    "metric": "mae",
    "verbosity": -1,
    "n_jobs": -1,
    "random_state": 42,

    'n_estimators': 4839,
    'learning_rate': 0.051513721757191794,
    'num_leaves': 1443,
    'max_depth': 15,
    'min_child_samples': 20,
    'reg_alpha': 5.63026372759641,
    'reg_lambda': 0.035867442198184704,
    'subsample': 0.8152220908526123,
    'subsample_freq': 4,
    'colsample_bytree': 0.9394439290272043
}

In [39]:
data_limit = 0
if data_limit > 0:
    X_np = X_np[:data_limit]
    y_np = y_np[:data_limit]

X_df = pd.DataFrame(X_np, columns=features)
split_idx = int(len(X_df) * 0.8)
X_train = X_df[:split_idx]
X_val = X_df[split_idx:]
y_train = y_np[:split_idx]
y_val = y_np[split_idx:]

boostEnable = False
if boostEnable:
    model = lgb.LGBMRegressor(**params_bstg)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="mae",
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(100)
        ]
    )

    print(model.best_iteration_)
    preds = model.predict(X_val, num_iteration=model.best_iteration_)
    mae = mean_absolute_error(y_val, preds)
    print("MAE:", mae)

In [ ]:
features_to_drop = [
    "precipprob",
    "is_snow",
    "is_rain",
    "snow",
    "feelslike",
    "snowdepth"
]

X_df = pd.DataFrame(X_np, columns=features)
X_df = X_df.drop(columns=features_to_drop)
print(f"Удалены признаки: {features_to_drop}")
print(f"Осталось признаков: {X_df.shape[1]}")
print(f"Оставшиеся признаки: {list(X_df.columns)}")

In [41]:
if boostEnable:
    X_train = X_df[:split_idx]
    X_val = X_df[split_idx:]

    model = lgb.LGBMRegressor(**params_bstg)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="mae",
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(100)
        ]
    )

    print(model.best_iteration_)
    preds = model.predict(X_val, num_iteration=model.best_iteration_)
    mae = mean_absolute_error(y_val, preds)


    print(f"MAE с очищенными признаками: {mae:.4f}")

Удалены признаки: ['precipprob', 'is_snow', 'is_rain', 'snow', 'feelslike', 'snowdepth']
Осталось признаков: 25
Оставшиеся признаки: ['PULocationID', 'month', 'day', 'dayofweek', 'hour', 'trips_lag_1h', 'trips_lag_day', 'trips_lag_week', 'roll_mean_24h', 'roll_std_24h', 'roll_mean_week', 'hour_sin', 'hour_cos', 'zone_avg_trips', 'temp', 'dew', 'humidity', 'precip', 'windgust', 'windspeed', 'winddir', 'sealevelpressure', 'cloudcover', 'visibility', 'uvindex']


In [ ]:
PI_enable = False

In [ ]:
if PI_enable:

    def rfe_with_permutation(X_tr, X_v, y_tr, y_v, params, n_to_keep):

        current_X_tr = X_tr.copy()
        current_X_v = X_v.copy()

        with open("PI_columns.txt", "w", encoding="utf-8") as file:
            file.write(f"{'Осталось':<10} | {'MAE (Val)':<12} | {'Удаляемый (PI)'}\n")
            file.write(("-" * 60) + "\n")

        counter = 0

        while current_X_tr.shape[1] > n_to_keep:
            counter += 1

            params["n_estimators"] = 1000
            model = lgb.LGBMRegressor(
                **params
            )

            model.fit(
                current_X_tr, y_tr,
                eval_set=[(current_X_v, y_v)],
                eval_metric="mae",
                callbacks=[
                    lgb.early_stopping(stopping_rounds=50),
                    lgb.log_evaluation(100)
                ]
            )

            preds = model.predict(current_X_v, num_iteration=model.best_iteration_)
            score = mean_absolute_error(y_v, preds)

            r = permutation_importance(
                model,
                current_X_v,
                y_v,
                n_repeats=5,
                random_state=42,
                n_jobs=-1,
                scoring="neg_mean_absolute_error"
            )

            pi_importances = pd.Series(r.importances_mean, index=current_X_tr.columns)

            if counter == 1:
                print(pi_importances.sort_values())

            worst_feat = pi_importances.sort_values().index[0]

            if pi_importances.min() > 0:
                print("Stopping: no clearly useless features left.")
                break

            with open("PI_columns.txt", "a", encoding="utf-8") as file:
                file.write(f"{current_X_tr.shape[1]:<10} | {score:<12.4f} | {worst_feat}\n")

            current_X_tr = current_X_tr.drop(columns=[worst_feat])
            current_X_v = current_X_v.drop(columns=[worst_feat])

        return current_X_tr.columns.tolist()

    best_features = rfe_with_permutation(
        X_train,
        X_val,
        y_train,
        y_val,
        final_params_bstg,
        n_to_keep=5
    )

### Отбор признаков с помощью LightGBM

In [109]:
def lgbm_feature_selection_cv(X_np, y_np, feature_names, params, splitter, top_n=15):
    X_df_fs = pd.DataFrame(X_np, columns=feature_names)

    fold_maes = []
    fold_importances = []

    for fold, (train_idx, val_idx) in enumerate(splitter.split(X_np), start=1):
        X_tr = X_df_fs.iloc[train_idx]
        X_vl = X_df_fs.iloc[val_idx]
        y_tr = y_np[train_idx]
        y_vl = y_np[val_idx]

        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_vl, y_vl)],
            eval_metric="mae",
            callbacks=[
                lgb.early_stopping(stopping_rounds=100),
                lgb.log_evaluation(period=100)
            ]
        )

        preds = model.predict(X_vl, num_iteration=model.best_iteration_)
        fold_mae = mean_absolute_error(y_vl, preds)
        fold_maes.append(fold_mae)

        fold_importances.append(
            pd.DataFrame({
                "feature": feature_names,
                f"importance_fold_{fold}": model.feature_importances_
            })
        )

        print(f"Fold {fold}: MAE = {fold_mae:.4f}, best_iteration = {model.best_iteration_}")

    feature_importances = fold_importances[0]
    for imp_df in fold_importances[1:]:
        feature_importances = feature_importances.merge(imp_df, on="feature", how="left")

    importance_cols = [c for c in feature_importances.columns if c.startswith("importance_fold_")]
    feature_importances["importance_mean"] = feature_importances[importance_cols].mean(axis=1)
    feature_importances = feature_importances.sort_values("importance_mean", ascending=False).reset_index(drop=True)

    selected_features = feature_importances.head(top_n)["feature"].tolist()

    print(f"CV MAE mean: {np.mean(fold_maes):.4f}")
    print(f"CV MAE std:  {np.std(fold_maes):.4f}")
    print(f"Selected top {top_n} features:")
    print(selected_features)

    return feature_importances, selected_features, fold_maes


top_n = 15
feature_importances, selected_features, fold_maes = lgbm_feature_selection_cv(
    X_np=X_np,
    y_np=y_np,
    feature_names=features,
    params=params_bstg,
    splitter=tscv,
    top_n=top_n,
)

display(feature_importances[["feature", "importance_mean"] + [c for c in feature_importances.columns if c.startswith("importance_fold_")]])

Training until validation scores don't improve for 100 rounds
[100]	valid_0's l1: 12.2086
[200]	valid_0's l1: 11.8196
[300]	valid_0's l1: 11.5298
[400]	valid_0's l1: 11.3389
[500]	valid_0's l1: 11.1469
[600]	valid_0's l1: 10.9866
[700]	valid_0's l1: 10.8882
[800]	valid_0's l1: 10.7992
[900]	valid_0's l1: 10.7338
[1000]	valid_0's l1: 10.6677
[1100]	valid_0's l1: 10.5958
[1200]	valid_0's l1: 10.5267
[1300]	valid_0's l1: 10.4823
[1400]	valid_0's l1: 10.437
[1500]	valid_0's l1: 10.4232
[1600]	valid_0's l1: 10.3916
[1700]	valid_0's l1: 10.3656
[1800]	valid_0's l1: 10.3513
[1900]	valid_0's l1: 10.3238
[2000]	valid_0's l1: 10.3083
[2100]	valid_0's l1: 10.2805
[2200]	valid_0's l1: 10.2563
[2300]	valid_0's l1: 10.2496
[2400]	valid_0's l1: 10.2311
[2500]	valid_0's l1: 10.2165
[2600]	valid_0's l1: 10.2114
[2700]	valid_0's l1: 10.1948
[2800]	valid_0's l1: 10.1897
[2900]	valid_0's l1: 10.1769
[3000]	valid_0's l1: 10.1709
[3100]	valid_0's l1: 10.1685
[3200]	valid_0's l1: 10.1595
[3300]	valid_0's l1:

,feature,importance_mean,importance_fold_1,importance_fold_2,importance_fold_3
0,trips_lag_day,309701.000000,547593,283730,97780
1,trips_lag_week,296123.000000,522118,270129,96122
2,trips_lag_1h,295353.000000,513866,270838,101355
3,roll_std_24h,268823.000000,462058,249883,94528
4,PULocationID,262280.333333,441729,246769,98343
5,roll_mean_24h,233443.666667,397143,217247,85941
6,zone_avg_trips,201255.666667,330589,191266,81912
7,roll_mean_week,195973.000000,344805,179899,63215
8,winddir,134940.000000,230503,125916,48401
9,sealevelpressure,132392.000000,224786,124534,47856


### Подбор гиперпараметров

In [ ]:
def objective_boosting(trial):
    param = {
        "boosting_type": "gbdt",
        "objective": "regression_l1",
        "metric": "mae",
        "verbosity": -1,
        "n_jobs": -1,
        "random_state": 42,

        "n_estimators": trial.suggest_int("n_estimators", 4000, 8000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 500, 2000),
        "max_depth": trial.suggest_int("max_depth", 12, 17),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),

        "reg_alpha": trial.suggest_float("reg_alpha", 5.0, 25.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 10.0, log=True),

        "subsample": trial.suggest_float("subsample", 0.7, 0.95),
        "subsample_freq": 4,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.8, 1.0),
    }

    scores = []

    for i, (train_idx, val_idx) in enumerate(tscv.split(X_np)):
        X_train, X_val = X_np[train_idx], X_np[val_idx]
        y_train, y_val = y_np[train_idx], y_np[val_idx]

        model = lgb.LGBMRegressor(**param)

        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(stopping_rounds=50),
                lgb.log_evaluation(period=0)
            ]
        )

        preds = model.predict(X_val)
        mae = mean_absolute_error(y_val, preds)
        scores.append(mae)

        trial.report(mae, i)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(scores)


study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1)
)


study.optimize(objective_boosting, n_trials=20)
print(f"Лучшие параметры: {study.best_params}")

### Визуализация важности гиперпараметров при подборе

In [ ]:
vis.plot_param_importances(study)

## Алгоритм Random Forest

### Подбор гиперпараметров

In [37]:
def objective(trial):
    param = {
        "boosting_type": "rf",
        "objective": "regression_l1",
        "metric": "mae",
        "n_estimators": 1189,
        "num_leaves": 3759,
        "max_depth": 29,
        "min_child_samples": 247,
        "subsample": 0.9643311966144387,
        "subsample_freq": 1,
        "feature_fraction":  0.47817161713615186,
        "n_jobs": -1,
        "random_state": 42,
        "verbose": -1,
        "device": "gpu"
    }

    scores = []

    for i, (train_idx, val_idx) in enumerate(tscv.split(X_np)):
        X_train, X_val = X_np[train_idx], X_np[val_idx]
        y_train, y_val = y_np[train_idx], y_np[val_idx]

        model = lgb.LGBMRegressor(**param)

        model.fit(X_train, y_train)
        preds = model.predict(X_val)

        mae = mean_absolute_error(y_val, preds)
        scores.append(mae)

        trial.report(mae, i)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(scores)

study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1)
)
study.optimize(objective, n_trials=20)
print(f"Лучшие параметры: {study.best_params}")

[I 2026-05-05 11:15:19,230] A new study created in memory with name: no-name-8ef1ae2f-34f2-4d4c-beec-2e3973db7f3f


### Визуализация важности гиперпараметров при подборе

In [38]:
vis.plot_param_importances(study)

[W 2026-05-05 11:15:19,251] Your study does not have any completed trials.


In [ ]:
final_params_rf = {
    "boosting_type": "rf",
    "objective": "regression_l1",
    "metric": "mae",
    "n_estimators": 1189,
    "num_leaves": 3759,
    "max_depth": 29,
    "min_child_samples": 247,
    "subsample": 0.9643311966144387,
    "subsample_freq": 1,
    "feature_fraction":  0.47817161713615186,
    "n_jobs": -1,
    "random_state": 42,
    "verbose": -1,
}

## LSTM

### Обучение модели

In [61]:
class TaxiHybridDataset(Dataset):
    def __init__(self, X, y, window_size=12):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.window_size = window_size

    def __len__(self):
        return len(self.X) - self.window_size

    def __getitem__(self, idx):
        # Последовательность для LSTM (окно)
        seq = self.X[idx : idx + self.window_size]
        # Статические признаки текущего момента (последняя строка окна)
        # Здесь уже сидят твои lag_1d и lag_1w
        static = self.X[idx + self.window_size - 1] 
        target = self.y[idx + self.window_size - 1]
        return seq, static, target

In [62]:
class TaxiHybridModel(nn.Module):
    def __init__(self, input_size, static_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        
        # Вход в финальный слой: выход LSTM + все статические признаки (вкл. лаги)
        self.fc = nn.Linear(hidden_size + static_size, 64)
        self.out = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, seq, static):
        lstm_out, _ = self.lstm(seq)
        last_hidden = lstm_out[:, -1, :] # Берем состояние последнего часа
        
        # Склеиваем "динамику" из LSTM и "факты" (лаги за день/неделю)
        combined = torch.cat((last_hidden, static), dim=1)
        
        x = self.relu(self.fc(combined))
        return self.out(x)

In [63]:
split_idx = int(len(X_np) * 0.8)

X_train_raw = X_np[:split_idx]
X_val_raw   = X_np[split_idx:]

y_train_raw = y_np[:split_idx]
y_val_raw   = y_np[split_idx:]

from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train = scaler_X.fit_transform(X_train_raw)
X_val   = scaler_X.transform(X_val_raw)

y_train = scaler_y.fit_transform(y_train_raw.reshape(-1, 1)).ravel()
y_val   = scaler_y.transform(y_val_raw.reshape(-1, 1)).ravel()

In [66]:
def objective(trial):

    device = torch.device("cuda")

    # --- HYPERPARAMS ---
    win_size = trial.suggest_int("window_size", 6, 24)
    hidden_size = trial.suggest_int("hidden_size", 64, 256)
    num_layers = trial.suggest_int("num_layers", 1, 2)
    lr = trial.suggest_float("lr", 1e-4, 5e-4, log=True)
    batch_size = trial.suggest_categorical("batch_size", [256, 512])

    # --- DATA ---
    train_ds = TaxiHybridDataset(X_train, y_train, window_size=win_size)
    val_ds   = TaxiHybridDataset(X_val, y_val, window_size=win_size)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=True)

    # --- MODEL ---
    model = TaxiHybridModel(
        input_size=X_train.shape[1],
        static_size=X_train.shape[1],
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=0.2
    ).to(device)

    if torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(model)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.L1Loss()

    # --- EARLY STOPPING ---
    best_val = float("inf")
    patience = 3
    counter = 0

    for epoch in range(20):

        # ===== TRAIN =====
        model.train()
        for seq, static, target in train_loader:
            seq = seq.to(device)
            static = static.to(device)
            target = target.to(device)

            optimizer.zero_grad()

            output = model(seq, static)
            loss = criterion(output.squeeze(), target)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()

        # ===== VALIDATION =====
        model.eval()
        val_losses = []

        with torch.no_grad():
            for seq, static, target in val_loader:
                seq = seq.to(device)
                static = static.to(device)
                target = target.to(device)

                output = model(seq, static)

                output_real = scaler_y.inverse_transform(
                    output.detach().cpu().numpy().reshape(-1, 1)
                )
                
                target_real = scaler_y.inverse_transform(
                    target.detach().cpu().numpy().reshape(-1, 1)
                )
                
                loss = mean_absolute_error(target_real, output_real)

                val_losses.append(loss)

        val_mae = np.mean(val_losses)
        print(f"Epoch {epoch} | Val MAE: {val_mae}")

        # ===== EARLY STOPPING =====
        if val_mae < best_val:
            best_val = val_mae
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            break

    return best_val

In [ ]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print("Best params:", study.best_params)
print("Best MAE:", study.best_value)

[I 2026-05-05 11:44:59,188] A new study created in memory with name: no-name-3a471de3-393f-4773-8c31-d5dcb72028f6


Epoch 0 | Val MAE: 12.950993317509663
Epoch 1 | Val MAE: 12.087196797759665
Epoch 2 | Val MAE: 11.861678332345893
Epoch 3 | Val MAE: 11.848418366044923
Epoch 4 | Val MAE: 11.770908513295865
Epoch 5 | Val MAE: 11.823257795461712
Epoch 6 | Val MAE: 12.021247291416278


[I 2026-05-05 11:52:36,909] Trial 0 finished with value: 11.770908513295865 and parameters: {'window_size': 12, 'hidden_size': 97, 'num_layers': 1, 'lr': 0.0001339595470549919, 'batch_size': 256}. Best is trial 0 with value: 11.770908513295865.


Epoch 7 | Val MAE: 12.011759380019464
Epoch 0 | Val MAE: 12.987827065765904
Epoch 1 | Val MAE: 12.11128662883167
Epoch 2 | Val MAE: 11.76422710578514
Epoch 3 | Val MAE: 12.099522011036624
Epoch 4 | Val MAE: 12.24989282518091


[I 2026-05-05 11:58:45,450] Trial 1 finished with value: 11.76422710578514 and parameters: {'window_size': 17, 'hidden_size': 130, 'num_layers': 1, 'lr': 0.0002474081913017115, 'batch_size': 256}. Best is trial 1 with value: 11.76422710578514.


...

Epoch 4 | Val MAE: 11.983637654400392
Epoch 0 | Val MAE: 12.88483742612916
Epoch 1 | Val MAE: 12.65546829901009
Epoch 2 | Val MAE: 12.280428213493847
Epoch 3 | Val MAE: 11.687791755266279
Epoch 4 | Val MAE: 11.808005700601596
Epoch 5 | Val MAE: 11.641639367813633
Epoch 6 | Val MAE: 11.523725919263013
Epoch 7 | Val MAE: 11.49628409344088
Epoch 8 | Val MAE: 11.574796619073624
Epoch 9 | Val MAE: 12.499453286886958


[I 2026-05-05 12:18:42,903] Trial 4 finished with value: 11.49628409344088 and parameters: {'window_size': 12, 'hidden_size': 167, 'num_layers': 1, 'lr': 0.00028964828326206397, 'batch_size': 512}. Best is trial 4 with value: 11.49628409344088.


Epoch 10 | Val MAE: 11.578902442135915
Epoch 0 | Val MAE: 12.191069682439169
Epoch 1 | Val MAE: 12.332897635635186
Epoch 2 | Val MAE: 12.305425599356678
Epoch 3 | Val MAE: 12.005180463612637
Epoch 4 | Val MAE: 11.677891818905174
Epoch 5 | Val MAE: 11.545616913064618
Epoch 6 | Val MAE: 11.52518514680714
Epoch 7 | Val MAE: 11.878372752035148
Epoch 8 | Val MAE: 11.6218102650479


[I 2026-05-05 12:25:23,524] Trial 5 finished with value: 11.52518514680714 and parameters: {'window_size': 11, 'hidden_size': 111, 'num_layers': 2, 'lr': 0.0003140264989036354, 'batch_size': 512}. Best is trial 4 with value: 11.49628409344088.


Epoch 9 | Val MAE: 12.296560296759798
Epoch 0 | Val MAE: 13.22879616716569
Epoch 1 | Val MAE: 12.165466844849869
Epoch 2 | Val MAE: 11.878391691086076
Epoch 3 | Val MAE: 11.705430775787972
Epoch 4 | Val MAE: 11.50905236054061
Epoch 5 | Val MAE: 11.318475613712893
Epoch 6 | Val MAE: 11.584997816249217


# Оценка построенных моделей на отложенной выборке